In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import importlib
import gc
import io
import os
from itertools import combinations

from IPython.display import display

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)

pd.reset_option('display.float_format')
pd.set_option('display.max_colwidth', None)

from config import ROOT, prev_num_aggregations  # lib này được khởi tạo ban đầu dự án

import helpers.view as view
import helpers.EDA as EDA
import modules.utils as utils

importlib.reload(view)
importlib.reload(EDA)
importlib.reload(utils)

from helpers.cache_clear import cache_clear

get_pickle = utils.get_pickle
get_pickles = utils.get_pickles

In [3]:
prev = pd.read_pickle(utils.get_pickle_paths("prev")[0])

In [13]:
KEY = "SK_ID_CURR"

In [7]:
cols = [KEY, 'NAME_CONTRACT_STATUS', 'AMT_ANNUITY','DAYS_LAST_DUE_1ST_VERSION','DAYS_DECISION', 'active', 'completed', 'total_debt', 'amt_paid', 'amt_unpaid', 'cnt_paid', 'cnt_unpaid']
cols += prev.filter(regex='^future_payment_').columns.tolist()
cols += prev.filter(regex='^past_payment_').columns.tolist()

In [9]:
prev = utils.get_pickles("prev", cols)

In [33]:
base = prev[[KEY]].drop_duplicates().set_index(KEY)

In [30]:
gr = prev.groupby(KEY)
gr_app = prev[prev['NAME_CONTRACT_STATUS']=='Approved'].groupby(KEY)
gr_ref = prev[prev['NAME_CONTRACT_STATUS']=='Refused'].groupby(KEY)
gr_act = prev[prev['active']==1].groupby(KEY)
gr_cmp = prev[prev['completed']==1].groupby(KEY)

In [10]:
app_cols = ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 
       'AMT_CREDIT-d-AMT_ANNUITY', 'DAYS_BIRTH']

In [20]:
train = utils.get_pickles("train", [KEY] + app_cols)
test = utils.get_pickles("test", [KEY] + app_cols)

In [22]:
train["AMT_ANNUITY"] = train["AMT_ANNUITY"].fillna(0)
test["AMT_ANNUITY"] = test["AMT_ANNUITY"].fillna(0)

In [24]:
train.columns = [KEY] + ['app_'+c for c in train.columns[1:]]
test.columns  = [KEY] + ['app_'+c for c in test.columns[1:]]

In [26]:
col_init = train.columns.tolist()

In [38]:
base['approved_cnt']  = gr_app.size()
base['resfused_cnt']  = gr_ref.size()
base['active_cnt']  = gr_act.size()
base['completed_cnt']  = gr_cmp.size()

base['app-p-ref_cnt'] = base['approved_cnt'] + base['resfused_cnt']
base['approved_ratio'] = base['approved_cnt'] / base['app-p-ref_cnt']
base['active_ratio']   = base['active_cnt'] / base['approved_cnt']

In [39]:
c = 'total_debt'
base[f'{c}_min']  = gr_app[c].min()
base[f'{c}_mean'] = gr_app[c].mean()
base[f'{c}_max']  = gr_app[c].max()
base[f'{c}_sum'] = gr_app[c].sum()

c = 'amt_paid'
base[f'{c}_min']  = gr_app[c].min()
base[f'{c}_mean'] = gr_app[c].mean()
base[f'{c}_max']  = gr_app[c].max()
base[f'{c}_sum'] = gr_app[c].sum()

c = 'amt_unpaid'
base[f'{c}_min']  = gr_app[c].min()
base[f'{c}_mean'] = gr_app[c].mean()
base[f'{c}_max']  = gr_app[c].max()
base[f'{c}_sum'] = gr_app[c].sum()

c = 'cnt_paid'
base[f'{c}_min']  = gr_app[c].min()
base[f'{c}_mean'] = gr_app[c].mean()
base[f'{c}_max']  = gr_app[c].max()
base[f'{c}_sum'] = gr_app[c].sum()

c = 'cnt_unpaid'
base[f'{c}_min']  = gr_app[c].min()
base[f'{c}_mean'] = gr_app[c].mean()
base[f'{c}_max']  = gr_app[c].max()
base[f'{c}_sum'] = gr_app[c].sum()

In [40]:
# ratio
base['amt_paid-d-unpaid_min']  = base['amt_paid_min']  / base['amt_unpaid_min']
base['amt_paid-d-unpaid_mean'] = base['amt_paid_mean'] / base['amt_unpaid_mean']
base['amt_paid-d-unpaid_max']  = base['amt_paid_max']  / base['amt_unpaid_max']
base['amt_paid-d-unpaid_sum']  = base['amt_paid_sum']  / base['amt_unpaid_sum']

base['cnt_paid-d-unpaid_min']  = base['cnt_paid_min']  / base['cnt_unpaid_min']
base['cnt_paid-d-unpaid_mean'] = base['cnt_paid_mean'] / base['cnt_unpaid_mean']
base['cnt_paid-d-unpaid_max']  = base['cnt_paid_max']  / base['cnt_unpaid_max']
base['cnt_paid-d-unpaid_sum']  = base['cnt_paid_sum']  / base['cnt_unpaid_sum']

In [43]:
c = 'AMT_ANNUITY'
base[f'{c}_act_min']  = gr_act[c].min()
base[f'{c}_act_mean'] = gr_act[c].mean()
base[f'{c}_act_max']  = gr_act[c].max()
base[f'{c}_act_sum'] = gr_act[c].sum()

base[f'{c}_cmp_min']  = gr_cmp[c].min()
base[f'{c}_cmp_mean'] = gr_cmp[c].mean()
base[f'{c}_cmp_max']  = gr_cmp[c].max()
base[f'{c}_cmp_sum'] = gr_cmp[c].sum()

c = 'DAYS_LAST_DUE_1ST_VERSION'
base[f'{c}_act_min']  = gr_act[c].min()
base[f'{c}_act_mean'] = gr_act[c].mean()
base[f'{c}_act_max']  = gr_act[c].max()
base[f'{c}_act_sum'] = gr_act[c].sum()



base['DAYS_DECISION_min'] = gr['DAYS_DECISION'].min()
base['DAYS_DECISION_max'] = gr['DAYS_DECISION'].max()

base['amt_paid_sum-d-total_debt_sum'] = base['amt_paid_sum'] / base['total_debt_sum']
base['amt_paid_sum-d-amt_unpaid_sum'] = base['amt_paid_sum'] / base['amt_unpaid_sum']

base['DAYS_DECISION_app_min'] = gr_app['DAYS_DECISION'].min()
base['DAYS_DECISION_app_max'] = gr_app['DAYS_DECISION'].max()

base['DAYS_DECISION_ref_min'] = gr_ref['DAYS_DECISION'].min()
base['DAYS_DECISION_ref_max'] = gr_ref['DAYS_DECISION'].max()

In [ ]:
# future payment
col = prev.head().filter(regex='^future_payment_').columns
col_future_sum  = []
col_future_min  = []
col_future_mean = []
col_future_max  = []
for c in col:
    base[f'{c}_sum'] = gr_act[c].sum()
    col_future_sum.append(f'{c}_sum')
    base[f'{c}_min'] = gr_act[c].min()
    col_future_min.append(f'{c}_min')
    base[f'{c}_max'] = gr_act[c].max()
    col_future_max.append(f'{c}_max')
    base[f'{c}_mean'] = gr_act[c].mean()
    col_future_mean.append(f'{c}_mean')

# past payment
col = prev.head().filter(regex='^past_payment_').columns
col_past_sum = []
col_past_min = []
col_past_mean = []
col_past_max = []
for c in col:
    base[f'{c}_sum'] = gr_app[c].sum()
    col_past_sum.append(f'{c}_sum')
    base[f'{c}_min'] = gr_app[c].min()
    col_past_min.append(f'{c}_min')
    base[f'{c}_mean'] = gr_app[c].mean()
    col_past_mean.append(f'{c}_mean')
    base[f'{c}_max'] = gr_app[c].max()
    col_past_max.append(f'{c}_max')

In [46]:
base.reset_index(inplace=True)

C:\Users\Admin\AppData\Local\Temp\ipykernel_13036\3777416534.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  base.reset_index(inplace=True)


In [48]:
def make_feature(df):
    
    df['total_debt_sum-p-app']            = df['total_debt_sum'] + df['app_AMT_CREDIT']
    df['total_debt_sum-p-app-d-income'] = df['total_debt_sum-p-app'] / df['app_AMT_INCOME_TOTAL']
    df['amt_unpaid_sum-p-app']            = df['amt_unpaid_sum'] + df['app_AMT_CREDIT']
    df['amt_unpaid_sum-p-app-d-income'] = df['amt_unpaid_sum-p-app'] / df['app_AMT_INCOME_TOTAL']

    # future payment
    df[col_future_sum+col_past_sum] = df[col_future_sum+col_past_sum].fillna(0)
    tmp = np.ceil(df['app_AMT_CREDIT-d-AMT_ANNUITY']).clip(lower=0)

    # số tiền phải trả trong các tháng tương lai = bao gồm tiền từ prev chưa trả hết + số tiền từ khoản vay hiện tại
    col = []
    for i, c in enumerate(col_future_sum, 1):
        c1 = f'prevapp_future_payment_{i}m'
        df[c1] = df[c] + np.where(tmp >= 1, df['app_AMT_ANNUITY'], 0)
        tmp -= 1
        col.append(c1)
    
    
    
    df['prevapp_future_payment_1vs2'] = df['prevapp_future_payment_1m'] / df['prevapp_future_payment_2m']
    df['prevapp_future_payment_1vs3'] = df['prevapp_future_payment_1m'] / df['prevapp_future_payment_3m']
    df['prevapp_future_payment_1vs4'] = df['prevapp_future_payment_1m'] / df['prevapp_future_payment_4m']
    df['prevapp_future_payment_1vs5'] = df['prevapp_future_payment_1m'] / df['prevapp_future_payment_5m']
    df['prevapp_future_payment_1vs6'] = df['prevapp_future_payment_1m'] / df['prevapp_future_payment_6m']
    
    
    
    # future
    df['prevapp_future_payment_max'] = df[col].max(1) # next month total
    df['prev_future_payment_max'] = df['prevapp_future_payment_max'] - df['app_AMT_ANNUITY'] # without app
    df['future_payment_app_ratio'] = df['prevapp_future_payment_max'] / df['prev_future_payment_max']
    
    df['prevapp_future_payment_max-d-income']  = df['prevapp_future_payment_max'] / df['app_AMT_INCOME_TOTAL']
    df['prevapp_future_payment_max-d-credit']  = df['prevapp_future_payment_max'] / df['app_AMT_CREDIT']
    df['prevapp_future_payment_max-d-annuity'] = df['prevapp_future_payment_max'] / df['app_AMT_ANNUITY']
    df['prev_future_payment_max-d-income']     = df['prev_future_payment_max']    / df['app_AMT_INCOME_TOTAL']
    df['prev_future_payment_max-d-credit']     = df['prev_future_payment_max']    / df['app_AMT_CREDIT']
    df['prev_future_payment_max-d-annuity']    = df['prev_future_payment_max']    / df['app_AMT_ANNUITY']
    
    
    
    # past
    df['past_payment_sum_max'] = df[col_past_sum].max(1) # past max
    df['past_payment_sum_max-d-income'] = df['past_payment_sum_max'] / df['app_AMT_INCOME_TOTAL']
    df['past_payment_sum_max-d-credit'] = df['past_payment_sum_max'] / df['app_AMT_CREDIT']
    df['past_payment_sum_max-d-annuity'] = df['past_payment_sum_max'] / df['app_AMT_ANNUITY']
    
    
    
    # future vs past
    df['future_vs_past_max'] = df['prevapp_future_payment_max'] / df['past_payment_sum_max']
    df['future_vs_past_max_withoutapp'] = df['prev_future_payment_max'] / df['past_payment_sum_max']
    df['future_vs_past_max-vs-withoutapp'] = df['future_vs_past_max'] / df['future_vs_past_max_withoutapp']
    df['future_vs_past_max-d-income'] = df['prevapp_future_payment_max-d-income'] / df['past_payment_sum_max-d-income']
    
    df['DAYS_DECISION_min-m-DAYS_BIRTH'] = df['DAYS_DECISION_min'] - df['app_DAYS_BIRTH']
    df['DAYS_DECISION_max-m-DAYS_BIRTH'] = df['DAYS_DECISION_max'] - df['app_DAYS_BIRTH']

In [50]:
_train = pd.merge(train, base, on=KEY, how='left')
make_feature(_train)


_test = pd.merge(test, base, on=KEY, how='left')
make_feature(_test)

C:\Users\Admin\AppData\Local\Temp\ipykernel_13036\2319556176.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[c1] = df[c] + np.where(tmp >= 1, df['app_AMT_ANNUITY'], 0)
C:\Users\Admin\AppData\Local\Temp\ipykernel_13036\2319556176.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[c1] = df[c] + np.where(tmp >= 1, df['app_AMT_ANNUITY'], 0)
C:\Users\Admin\AppData\Local\Temp\ipykernel_13036\2319556176.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many tim

In [52]:
_train.drop(col_init, axis=1, inplace=True)
_test.drop(col_init, axis=1, inplace=True)

In [53]:
PREF = "f105_"

utils.to_feature(_train.add_prefix(PREF), name="train")
utils.to_feature(_test.add_prefix(PREF), name="test")

d:\Data Science/data/feature/train/f105_approved_cnt.f
d:\Data Science/data/feature/train/f105_resfused_cnt.f
d:\Data Science/data/feature/train/f105_active_cnt.f
d:\Data Science/data/feature/train/f105_completed_cnt.f
d:\Data Science/data/feature/train/f105_app-p-ref_cnt.f
d:\Data Science/data/feature/train/f105_approved_ratio.f
d:\Data Science/data/feature/train/f105_active_ratio.f
d:\Data Science/data/feature/train/f105_total_debt_min.f
d:\Data Science/data/feature/train/f105_total_debt_mean.f
d:\Data Science/data/feature/train/f105_total_debt_max.f
d:\Data Science/data/feature/train/f105_total_debt_sum.f
d:\Data Science/data/feature/train/f105_amt_paid_min.f
d:\Data Science/data/feature/train/f105_amt_paid_mean.f
d:\Data Science/data/feature/train/f105_amt_paid_max.f
d:\Data Science/data/feature/train/f105_amt_paid_sum.f
d:\Data Science/data/feature/train/f105_amt_unpaid_min.f
d:\Data Science/data/feature/train/f105_amt_unpaid_mean.f
d:\Data Science/data/feature/train/f105_amt_unpa